[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/exercices/seance2_exercices.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- transformer des colonnes de texte en variables utilisables par un modèle
- ajuster une régression logistique et lire une probabilité de départ
- expliquer pourquoi la justesse est un piège sur des données déséquilibrées
- lire une matrice de confusion, la précision et le rappel
- choisir un seuil de décision à partir d'un coût, pas d'une habitude

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")   ## texte -> nombre
tel = tel.dropna(subset=["total"])   ## 11 abonnes tout neufs, sans facture

y = tel["churn"]   ## 1 = il est parti, 0 = il est reste
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True)   ## texte -> 0/1

# stratify=y : le meme taux de resiliation des deux cotes du decoupage
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(len(X_tr), "abonnes d'apprentissage,", len(X_te), "de test")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le contrat, déjà

> **Votre mission :**
> - Calculer le taux de résiliation par type de contrat, en % arrondi à 1 décimale → `par_contrat`.
> - Mettre celui des contrats mensuels dans `taux_mensuel`.

In [ ]:
par_contrat = (tel.groupby("____")["churn"].mean() * 100).round(1)
taux_mensuel = par_contrat["mensuel"]

print(par_contrat)

In [ ]:
verifier("1 - churn des contrats mensuels", taux_mensuel == 42.7,
         "groupby('contrat') puis mean() sur churn")

### Exercice 2 — Ajuster le modèle

> **Votre mission :**
> - Construire un pipeline `StandardScaler` puis `LogisticRegression(max_iter=1000)` → `m`, et l'ajuster sur l'apprentissage.
> - Récupérer les probabilités de départ du jeu de test → `proba`.

In [ ]:
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
m.fit(____, ____)

proba = m.predict_proba(X_te)[:, ____]
print(proba[:5].round(3))

In [ ]:
verifier("2a - nombre de probabilites", len(proba) == 2110, "on predit sur le jeu de test")
verifier("2b - ce sont bien des probabilites", proba.min() >= 0 and proba.max() <= 1,
         "la colonne d'indice 1 est la probabilite de depart")

### Exercice 3 — Le piège de la justesse

> **Votre mission :**
> - Calculer la justesse du modèle sur le test → `just_modele` (en %, 1 décimale).
> - Puis celle d'un modèle qui prédit que **personne** ne part → `just_nul`.
> - Combien de points le modèle gagne-t-il réellement ?

In [ ]:
pred = m.predict(X_te)

just_modele = round(100 * accuracy_score(y_te, pred), 1)
just_nul = round(100 * (1 - y_te.____()), 1)
print(just_modele, "% contre", just_nul, "% ->", round(just_modele - just_nul, 1), "points")

In [ ]:
verifier("3a - justesse du modele", abs(just_modele - 79.8) < 0.5, "accuracy_score(y_te, pred)")
verifier("3b - justesse du modele nul", just_nul == 73.4,
         "c'est la proportion de clients qui restent")

### Exercice 4 — La matrice de confusion

> **Votre mission :**
> - Afficher la matrice de confusion du modèle.
> - Mettre dans `perdus` le nombre de clients **partis sans qu'on les ait détectés**.
> - C'est la case qui coûte de l'argent.

In [ ]:
mat = confusion_matrix(y_te, pred)
print(mat)

perdus = mat[1][____]
print(perdus, "clients perdus sans rien tenter")

In [ ]:
verifier("4 - clients perdus non detectes", abs(perdus - 255) <= 5,
         "ligne des vrais partants, colonne des predits restants")

### Exercice 5 — Précision et rappel

> **Votre mission :**
> - Calculer la précision → `prec` et le rappel → `rapp` du modèle, arrondis à 3 décimales.
> - Traduire chacun en une phrase de gestion, en commentaire.

In [ ]:
prec = round(precision_score(y_te, pred), 3)
rapp = round(____(y_te, pred), 3)

print("precision", prec, "| rappel", rapp)

In [ ]:
verifier("5a - precision", abs(prec - 0.640) < 0.02, "precision_score")
verifier("5b - rappel", abs(rapp - 0.545) < 0.02, "la fonction s'appelle recall_score")

### Exercice 6 — Descendre le seuil

> **Votre mission :**
> - Décider à **0,30** au lieu de 0,50 → `pred30`.
> - Recalculer précision et rappel → `prec30` et `rapp30`.
> - Lequel des deux monte, lequel descend ?

In [ ]:
pred30 = (proba > ____).astype(int)

prec30 = round(precision_score(y_te, pred30), 3)
rapp30 = round(recall_score(y_te, pred30), 3)
print("seuil 0.30 -> precision", prec30, "| rappel", rapp30)

In [ ]:
verifier("6a - rappel a 0,30", rapp30 > rapp, "un seuil plus bas retrouve plus de partants")
verifier("6b - precision a 0,30", prec30 < prec, "et contacte plus de monde pour rien")

### Exercice 7 — Le gain de la campagne

> **Votre mission :**
> - Un appel coûte **15 €**, un client retenu rapporte **300 €**, une relance en convainc **30 %**.
> - Calculer le gain au seuil 0,50 → `gain50`, puis au seuil 0,20 → `gain20`.

In [ ]:
def gain(seuil):
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()
    return vrais * 0.30 * 300 - p.sum() * ____

gain50 = gain(0.50)
gain20 = gain(0.20)
print(round(gain50), "euros contre", round(gain20), "euros")

In [ ]:
verifier("7a - gain au seuil 0,50", abs(gain50 - 20370) < 600, "le cout d'un appel est 15")
verifier("7b - gain au seuil 0,20", abs(gain20 - 28365) < 600, "meme calcul, seuil 0.20")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - Le directeur de la relation client vous demande combien d'appels prévoir et ce que ça rapporte.
> - Au seuil retenu de 0,20 : le nombre d'appels → `nb_appels`, et le gain par appel → `gain_appel` (arrondi à 2 décimales).
> - Puis rédigez votre recommandation en commentaire.

In [ ]:
p20 = (proba > 0.20).astype(int)

nb_appels = int(p20.____())
gain_appel = round(gain20 / nb_appels, 2)
print(nb_appels, "appels |", gain_appel, "euros de gain par appel")

In [ ]:
verifier("8a - nombre d'appels", abs(nb_appels - 1073) < 40, "sum() compte les 1")
verifier("8b - gain par appel", abs(gain_appel - 26.4) < 2, "divisez le gain par le nombre d'appels")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Quand l'appel coûte plus cher

> **Votre mission :**
> - Refaire le calcul du gain avec un appel à **60 €** au lieu de 15 € — une visite commerciale plutôt qu'un appel.
> - Où passe l'optimum ? Qu'est-ce que ça dit du réglage d'un modèle ?

### Question 10 — Les coefficients de la logistique

> **Votre mission :**
> - Extraire les coefficients du modèle et les trier par valeur absolue décroissante.
> - Les variables ayant été mises à l'échelle, ils sont comparables entre eux.
> - Quelles sont les trois qui pèsent le plus, et dans quel sens ?
> - *Nouveau :* dans un pipeline, on accède à la dernière étape par `m[-1]`.

### Question 11 — L'AUC ne dépend pas du seuil

> **Votre mission :**
> - Vérifier que l'AUC est identique quel que soit le seuil, alors que la justesse, elle, change.
> - Pourquoi cette propriété rend-elle l'AUC pratique pour **comparer deux modèles** ?

### Question 12 — Un modèle plus simple fait-il pire ?

> **Votre mission :**
> - Ajuster un second modèle avec **trois variables seulement** : `anc`, `mensuel` et le contrat.
> - Comparer son AUC à celle du modèle complet.
> - La différence justifie-t-elle de collecter les onze autres colonnes ?

### Question 13 — Le déséquilibre, traité autrement

> **Votre mission :**
> - Réajuster la logistique avec `class_weight='balanced'`, qui donne plus de poids à la classe rare.
> - Comparer justesse, précision et rappel au modèle d'origine.
> - En quoi cet effet ressemble-t-il à celui du seuil ?

### Question 14 — Qui sont les faux positifs ?

> **Votre mission :**
> - Isoler les abonnés que le modèle prédit partants **à tort**.
> - Comparer leur profil (ancienneté, facture, contrat) à celui des vrais partants.
> - Ces appels sont-ils vraiment perdus ?

### Question 15 — Question de synthèse

> **Votre mission :**
> - Le comité hésite : faut-il investir dans un meilleur modèle, ou dans un meilleur ciblage ?
> - Chiffrez les deux pistes : le gain obtenu en passant du seuil 0,50 au seuil optimal, et celui qu'apporterait un modèle parfait (rappel de 1 sans faux positifs) au seuil 0,50.
> - Puis tranchez, en commentaire.